## Step 7 — assign k-means cluster id to blocks
**# of cells in notebook:** 1

**Purpose:** Assign each block a macromorphology value based on the result of a k-means analysis. In that analysis, we calculcated several variables for individual buildings based on nearest neighor analyses. The k-means analysis assigned buildings to a cluster. In this step, we assign the block the majoirty cluster value based on the cluster values of the buildings within it.       

**Input:**

- a geopackage with a buildings layer, with field `bkm2`, representing the cluster value. (The actual workflow for the k-means analysis will be in a diferent notebook)
- a geodatabase with the blocks layer from steps 5 and 6
  
**Output:** blocks layer with new fields `cluster` and `tie`

**Main logic:**

1. Convert building polygons to points, if the buildings input is polygon geometry. If the buildings input is already points, copy it directly.
2. Spatially join block attributes to the building points, so each building point gets the block OBJECTID it falls within.
3. Count valid `bkm2` values by block `OBJECTID`, using only `bkm2 = 1` and `bkm2 = 2`.
4. For each block, compare the counts of bkm2 = 1 and bkm2 = 2 to determine the majority class.
5. Write the majority value back to the blocks layer in the `cluster` field. `cluster = NULL` when either: (a) the block has no valid building points, or (b) the counts of `bkm2 = 1` and `bkm2 = 2` are tied. The `tie` field is set to `1` only for tied blocks, and `0` otherwise.

In [ ]:
import arcpy
import os
import time
import traceback
from collections import defaultdict, Counter

# ============================================================
# USER INPUTS
# ============================================================

buildings = r"E:\World Bank deliverbale 1\knn\buildings_points_knn_kmeans_broad_compact.gpkg\main.wb_buildings_knn_broad_compact_kmeans"

blocks = r"E:\World Bank deliverbale 1\_analysis\blocks\blocks.gdb\juba_blocks_20260415_small_utm36n"

# Building cluster field
building_cluster_field = "bkm2"

# New / updated fields in blocks
cluster_field = "cluster"
tie_field = "tie"

# What to write when there is no majority
# Recommended: None, which writes NULL
no_majority_value = None


# ============================================================
# SETTINGS
# ============================================================

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

scratch_gdb = arcpy.env.scratchGDB

blocks_tmp = os.path.join(scratch_gdb, "tmp_blocks_with_oid")
building_points_tmp = os.path.join(scratch_gdb, "tmp_building_points_for_block_join")
building_block_sj = os.path.join(scratch_gdb, "tmp_building_block_sj")

block_oid_tmp_field = "block_oid_tmp"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def msg(text):
    print(text)
    arcpy.AddMessage(text)


def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)


def check_exists(path, label):
    if not arcpy.Exists(path):
        raise FileNotFoundError(f"{label} does not exist:\n{path}")


def get_field_name(table, field_name):
    """
    Return actual field name using case-insensitive matching.
    """
    for f in arcpy.ListFields(table):
        if f.name.lower() == field_name.lower():
            return f.name
    return None


def field_exists(table, field_name):
    return get_field_name(table, field_name) is not None


def print_fields(table, label):
    msg(f"\nFields in {label}:")
    for f in arcpy.ListFields(table):
        msg(f"  {f.name} | {f.type}")


def add_long_field_if_needed(table, field_name, nullable=True):
    if not field_exists(table, field_name):
        arcpy.management.AddField(
            in_table=table,
            field_name=field_name,
            field_type="LONG",
            field_is_nullable="NULLABLE" if nullable else "NON_NULLABLE"
        )
        msg(f"  Added field: {field_name}")
    else:
        msg(f"  Field already exists: {field_name}")
        msg("  Existing values will be overwritten.")


# ============================================================
# MAIN SCRIPT
# ============================================================

try:
    t0 = time.time()

    msg("Starting block majority-cluster assignment...")

    # --------------------------------------------------------
    # Check inputs
    # --------------------------------------------------------

    check_exists(buildings, "buildings")
    check_exists(blocks, "blocks")

    buildings_desc = arcpy.Describe(buildings)
    blocks_desc = arcpy.Describe(blocks)

    msg(f"\nBuildings layer:")
    msg(f"  Path: {buildings}")
    msg(f"  Count: {arcpy.management.GetCount(buildings)[0]}")
    msg(f"  Shape type: {buildings_desc.shapeType}")
    msg(f"  Spatial reference: {buildings_desc.spatialReference.name}")

    msg(f"\nBlocks layer:")
    msg(f"  Path: {blocks}")
    msg(f"  Count: {arcpy.management.GetCount(blocks)[0]}")
    msg(f"  Shape type: {blocks_desc.shapeType}")
    msg(f"  OID field: {blocks_desc.OIDFieldName}")
    msg(f"  Spatial reference: {blocks_desc.spatialReference.name}")

    if blocks_desc.shapeType != "Polygon":
        raise ValueError(f"Blocks must be polygon, but they are: {blocks_desc.shapeType}")

    bkm2_actual = get_field_name(buildings, building_cluster_field)

    if bkm2_actual is None:
        print_fields(buildings, "buildings")
        raise RuntimeError(f"Could not find field '{building_cluster_field}' in buildings layer.")

    msg(f"\nUsing building cluster field: {bkm2_actual}")

    if buildings_desc.spatialReference.name != blocks_desc.spatialReference.name:
        msg("\nWARNING: buildings and blocks have different spatial references.")
        msg("For best results, project them to the same CRS before running this workflow.")
        msg(f"  Buildings CRS: {buildings_desc.spatialReference.name}")
        msg(f"  Blocks CRS:    {blocks_desc.spatialReference.name}")

    # --------------------------------------------------------
    # Clean temporary outputs
    # --------------------------------------------------------

    msg("\nCleaning temporary outputs...")

    delete_if_exists(blocks_tmp)
    delete_if_exists(building_points_tmp)
    delete_if_exists(building_block_sj)

    # --------------------------------------------------------
    # Copy blocks to scratch and preserve original block OBJECTID
    # --------------------------------------------------------

    msg("\nCreating temporary blocks layer with preserved block OBJECTID...")

    arcpy.management.CopyFeatures(
        in_features=blocks,
        out_feature_class=blocks_tmp
    )

    if field_exists(blocks_tmp, block_oid_tmp_field):
        arcpy.management.DeleteField(blocks_tmp, block_oid_tmp_field)

    arcpy.management.AddField(
        in_table=blocks_tmp,
        field_name=block_oid_tmp_field,
        field_type="LONG"
    )

    arcpy.management.CalculateField(
        in_table=blocks_tmp,
        field=block_oid_tmp_field,
        expression=f"!{blocks_desc.OIDFieldName}!",
        expression_type="PYTHON3"
    )

    msg(f"  Temporary blocks created: {blocks_tmp}")
    msg(f"  Original block OBJECTID preserved in: {block_oid_tmp_field}")

    # --------------------------------------------------------
    # Prepare building assignment points
    # --------------------------------------------------------
    # If the buildings layer is polygon, create inside points.
    # If it is already point, copy it to scratch.
    #
    # For polygon buildings, INSIDE is intentionally used so the
    # assignment point is constrained to fall within the building footprint.
    # --------------------------------------------------------

    msg("\nPreparing building assignment points...")

    if buildings_desc.shapeType == "Polygon":
        msg("  Buildings are polygons. Creating inside points constrained to building footprints...")

        arcpy.management.FeatureToPoint(
            in_features=buildings,
            out_feature_class=building_points_tmp,
            point_location="INSIDE"
        )

    elif buildings_desc.shapeType == "Point":
        msg("  Buildings are already points. Copying points to scratch...")

        arcpy.management.CopyFeatures(
            in_features=buildings,
            out_feature_class=building_points_tmp
        )

    else:
        raise ValueError(
            f"Buildings layer must be polygon or point, but it is: {buildings_desc.shapeType}"
        )

    building_point_count = int(arcpy.management.GetCount(building_points_tmp)[0])
    msg(f"  Building assignment point count: {building_point_count}")

    if not field_exists(building_points_tmp, bkm2_actual):
        print_fields(building_points_tmp, "building assignment points")
        raise RuntimeError(f"Building assignment points are missing field: {bkm2_actual}")

    # --------------------------------------------------------
    # Spatial join building points to blocks
    # --------------------------------------------------------

    msg("\nSpatial joining building points to blocks...")

    arcpy.analysis.SpatialJoin(
        target_features=building_points_tmp,
        join_features=blocks_tmp,
        out_feature_class=building_block_sj,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_ALL",
        match_option="INTERSECT"
    )

    sj_count = int(arcpy.management.GetCount(building_block_sj)[0])
    msg(f"  Spatial join output: {building_block_sj}")
    msg(f"  Spatial join count: {sj_count}")

    block_oid_join_actual = get_field_name(building_block_sj, block_oid_tmp_field)
    bkm2_sj_actual = get_field_name(building_block_sj, bkm2_actual)

    if block_oid_join_actual is None:
        print_fields(building_block_sj, "building-block spatial join output")
        raise RuntimeError(f"Spatial join output is missing field: {block_oid_tmp_field}")

    if bkm2_sj_actual is None:
        print_fields(building_block_sj, "building-block spatial join output")
        raise RuntimeError(f"Spatial join output is missing field: {bkm2_actual}")

    # --------------------------------------------------------
    # Count bkm2 values by block
    # --------------------------------------------------------

    msg("\nCounting bkm2 classes within each block...")

    block_counts = defaultdict(Counter)

    skipped_no_block = 0
    skipped_null_bkm2 = 0
    skipped_other_bkm2 = 0

    with arcpy.da.SearchCursor(building_block_sj, [block_oid_join_actual, bkm2_sj_actual]) as cursor:
        for block_oid, bkm2_value in cursor:

            # No joined block
            if block_oid is None or block_oid == -1:
                skipped_no_block += 1
                continue

            # Null cluster value
            if bkm2_value is None:
                skipped_null_bkm2 += 1
                continue

            try:
                bkm2_int = int(bkm2_value)
            except Exception:
                skipped_other_bkm2 += 1
                continue

            # Expected values are 1 and 2.
            # This keeps the script strict and avoids accidentally counting bad codes.
            if bkm2_int not in [1, 2]:
                skipped_other_bkm2 += 1
                continue

            block_counts[int(block_oid)][bkm2_int] += 1

    msg(f"  Blocks with at least one valid building cluster: {len(block_counts)}")
    msg(f"  Building points skipped because no block joined: {skipped_no_block}")
    msg(f"  Building points skipped because bkm2 was NULL: {skipped_null_bkm2}")
    msg(f"  Building points skipped because bkm2 was not 1 or 2: {skipped_other_bkm2}")

    # --------------------------------------------------------
    # Determine majority class and tie flag by block
    # --------------------------------------------------------

    msg("\nDetermining majority bkm2 class and tie flag for each block...")

    block_cluster = {}
    block_tie = {}

    majority_1 = 0
    majority_2 = 0
    ties = 0

    for block_oid, counts in block_counts.items():
        count_1 = counts.get(1, 0)
        count_2 = counts.get(2, 0)

        if count_1 > count_2:
            block_cluster[block_oid] = 1
            block_tie[block_oid] = 0
            majority_1 += 1

        elif count_2 > count_1:
            block_cluster[block_oid] = 2
            block_tie[block_oid] = 0
            majority_2 += 1

        else:
            # Equal nonzero counts of 1 and 2
            block_cluster[block_oid] = no_majority_value
            block_tie[block_oid] = 1
            ties += 1

    msg(f"  Blocks with majority class 1: {majority_1}")
    msg(f"  Blocks with majority class 2: {majority_2}")
    msg(f"  Blocks with tied class counts: {ties}")

    # --------------------------------------------------------
    # Add output fields to blocks if needed
    # --------------------------------------------------------

    msg(f"\nPreparing output fields in blocks...")

    add_long_field_if_needed(blocks, cluster_field, nullable=True)
    add_long_field_if_needed(blocks, tie_field, nullable=True)

    cluster_field_actual = get_field_name(blocks, cluster_field)
    tie_field_actual = get_field_name(blocks, tie_field)

    # --------------------------------------------------------
    # Write majority cluster and tie values back to blocks
    # --------------------------------------------------------

    msg("\nWriting majority cluster and tie values back to blocks...")

    blocks_oid_field = blocks_desc.OIDFieldName

    updated_majority = 0
    written_ties = 0
    blocks_with_no_buildings = 0

    with arcpy.da.UpdateCursor(blocks, [blocks_oid_field, cluster_field_actual, tie_field_actual]) as cursor:
        for block_oid, current_cluster, current_tie in cursor:
            block_oid_int = int(block_oid)

            if block_oid_int not in block_cluster:
                # No buildings in this block
                cursor.updateRow([block_oid, no_majority_value, 0])
                blocks_with_no_buildings += 1

            else:
                cluster_value = block_cluster[block_oid_int]
                tie_value = block_tie[block_oid_int]

                cursor.updateRow([block_oid, cluster_value, tie_value])

                if tie_value == 1:
                    written_ties += 1
                elif cluster_value is not None:
                    updated_majority += 1

    msg(f"  Blocks assigned majority cluster: {updated_majority}")
    msg(f"  Blocks written as tied: {written_ties}")
    msg(f"  Blocks with no buildings: {blocks_with_no_buildings}")

    # --------------------------------------------------------
    # Clean temporary outputs
    # --------------------------------------------------------

    msg("\nCleaning temporary outputs...")

    delete_if_exists(blocks_tmp)
    delete_if_exists(building_points_tmp)
    delete_if_exists(building_block_sj)

    elapsed = round((time.time() - t0) / 60, 2)
    msg(f"\nDone. Elapsed time: {elapsed} minutes")

except Exception as e:
    msg("\nSCRIPT FAILED.")
    msg(str(e))
    msg(traceback.format_exc())
    raise